In [ ]:
import pandas as pd
import numpy as np

## 3. Create hourly aggregated Dateframe
Create a new df with hourly timestamps in the range of the dataset and all unique H3 indices so we have a complete grid for analysis

In [ ]:
# TODO: Validate data set paths
taxi_data_processed = pd.read_csv('../data/processed/taxi_data_processed.csv')
poi_data_processed = pd.read_csv('../data/processed/poi_data_processed.csv')
weather_data_processed = pd.read_csv('../data/processed/weather_data_processed.csv')

In [ ]:
# Extract hour from Trip Start Timestamp and Trip End Timestamp for temporal analysis
taxi_data_processed['Pickup Hour'] = pd.to_datetime(taxi_data_processed['Trip Start Timestamp'], format='%Y-%m-%d %H:%M:%S').dt.floor('h')
taxi_data_processed['Dropoff Hour'] = pd.to_datetime(taxi_data_processed['Trip End Timestamp'], format='%Y-%m-%d %H:%M:%S').dt.floor('h')

# Get the range of timestamps in the dataset
min_timestamp = taxi_data_processed['Pickup Hour'].min()
max_timestamp = taxi_data_processed['Dropoff Hour'].max()

# Create a complete hourly timestamp range
hourly_timestamps = pd.date_range(start=min_timestamp, end=max_timestamp, freq='h')

# Get all unique H3 indices from both pickup and dropoff
unique_h3_indices = pd.unique(pd.concat([taxi_data_processed['h3_index_pickup'], taxi_data_processed['h3_index_dropoff']]))

# Create a complete grid of all combinations of hourly timestamps and unique H3 indices
complete_grid = pd.MultiIndex.from_product(
    [hourly_timestamps, unique_h3_indices], names=['hour', 'h3_index']).to_frame(index=False)

In [ ]:
# Define aggregations specifically for the Pickups
pickup_aggregations = {
    'Total_Trip_Start': ('Trip ID', 'count'),
    'Unique Taxis': ('Taxi ID', 'nunique'),
    'AvgTripSeconds': ('Trip Seconds', 'mean'),
    'AvgTripMiles': ('Trip Miles', 'mean'),
    'AvgFare': ('Trip Total', 'mean'),
    'MostCommonCompany': ('Company', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    'CompanyCount': ('Company', 'nunique'),
    'PickupLatitude': ('Pickup Centroid Latitude', 'mean'),
    'PickupLongitude': ('Pickup Centroid Longitude', 'mean')
}

# Define aggregations specifically for the Dropoffs
dropoff_aggregations = {
    'Total_Trip_End': ('Trip ID', 'count')
}

# Perform two separate groupbys
agg_pickups = taxi_data_processed.groupby(
    ['h3_index_pickup', 'Pickup Hour']
).agg(**pickup_aggregations).reset_index()

agg_dropoffs = taxi_data_processed.groupby(
    ['h3_index_dropoff', 'Dropoff Hour']
).agg(**dropoff_aggregations).reset_index()


# Merge the Pickup data into the complete grid
aggregated_grid = complete_grid.merge(
    agg_pickups, 
    left_on=['hour', 'h3_index'], 
    right_on=['Pickup Hour', 'h3_index_pickup'], 
    how='left'
).drop(columns=['Pickup Hour', 'h3_index_pickup'])

# Merge the Dropoff data into the complete grid
aggregated_grid = aggregated_grid.merge(
    agg_dropoffs, 
    left_on=['hour', 'h3_index'], 
    right_on=['Dropoff Hour', 'h3_index_dropoff'], 
    how='left'
).drop(columns=['Dropoff Hour', 'h3_index_dropoff'])

# Fill missing values with 0 for count columns
aggregated_grid['Total_Trip_Start'] = aggregated_grid['Total_Trip_Start'].fillna(0)
aggregated_grid['Total_Trip_End'] = aggregated_grid['Total_Trip_End'].fillna(0)
aggregated_grid['Unique Taxis'] = aggregated_grid['Unique Taxis'].fillna(0)
aggregated_grid['AvgTripSeconds'] = aggregated_grid['AvgTripSeconds'].fillna(0)
aggregated_grid['AvgTripMiles'] = aggregated_grid['AvgTripMiles'].fillna(0)
aggregated_grid['AvgFare'] = aggregated_grid['AvgFare'].fillna(0)
aggregated_grid['CompanyCount'] = aggregated_grid['CompanyCount'].fillna(0)
aggregated_grid['MostCommonCompany'] = aggregated_grid['MostCommonCompany'].fillna("")

In [ ]:
aggregated_grid.to_parquet(
    "../data/aggregated_grid.parquet"
)